In [3]:
import requests
from bs4 import BeautifulSoup
import urllib.request
import json
from openai import OpenAI
import os
import time
import asyncio
import aiohttp
import requests
import websocket
import json
import threading
import time
import base64
import zlib
import json
import re
import json
from collections import defaultdict
from difflib import SequenceMatcher
import re
from dotenv import load_dotenv
from langchain.agents import load_tools
from langchain.agents import initialize_agent, Tool, create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.agents import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain.agents import AgentExecutor
from langchain.utilities import SerpAPIWrapper

load_dotenv()
open_ai_key = os.getenv("OPENAI_APIKEY")
serp_api_key = os.getenv("SERP_APIKEY")

openai_client = OpenAI(
  api_key=open_ai_key,
)

id_to_team = {'Atlanta Hawks': '1', 'Boston Celtics': '2', 'Brooklyn Nets': '17', 'Charlotte Hornets': '30', 'Chicago Bulls': '4', 'Cleveland Cavaliers': '5', 'Dallas Mavericks': '6', 'Denver Nuggets': '7', 'Detroit Pistons': '8', 'Golden State Warriors': '9', 'Houston Rockets': '10', 'Indiana Pacers': '11', 'LA Clippers': '12', 'Los Angeles Lakers': '13', 'Memphis Grizzlies': '29', 'Miami Heat': '14', 'Milwaukee Bucks': '15', 'Minnesota Timberwolves': '16', 'New Orleans Pelicans': '3', 'New York Knicks': '18', 'Oklahoma City Thunder': '25', 'Orlando Magic': '19', 'Philadelphia 76ers': '20', 'Phoenix Suns': '21', 'Portland Trail Blazers': '22', 'Sacramento Kings': '23', 'San Antonio Spurs': '24', 'Toronto Raptors': '28', 'Utah Jazz': '26', 'Washington Wizards': '27'}

In [4]:
def process_play_by_play(play_data):
    key_moments = []

    for play in play_data:
        time = play['clock']['displayValue']
        description = play['text']
        
        # Check for scoring plays
        if 'makes' in description or 'misses' in description:
            key_moments.append({
                'time': time,
                'event': 'scoring',
                'description': description
            })
        # Check for fouls
        elif 'foul' in description:
            key_moments.append({
                'time': time,
                'event': 'foul',
                'description': description
            })
        # Check for turnovers
        elif 'turnover' in description:
            key_moments.append({
                'time': time,
                'event': 'turnover',
                'description': description
            })
        # Add more conditions as needed

    return key_moments

def generate_queries_with_gpt(key_moments, game_info, api_key):
    prompt = (
        f"We are watching a live NBA game between {game_info['team1']} and {game_info['team2']} on {game_info['date']}. "
        f"Here are the key moments from the game so far:\n\n"
    )

    for moment in key_moments:
        prompt += f"{moment['time']}: {moment['description']} (Event: {moment['event']})\n"

    prompt += (
        "\nBased on these key moments, please generate 3-5 stat-driven queries that focus on historical comparisons, "
        "team trends, and notable player achievements in past games. Avoid hypothetical or predictive questions. "
        "The queries should be actionable and reflect a broader understanding of NBA stats and trends. "
        "Include queries about the game so far and comparisons to prior games this season or previous seasons."
        "When referring to seasons, use the format '2021-22 season'."
        "Avoid 'how' based qualitative questions and generate quantificable queries that can be answered with statistical data."
        "Make the queries specific, avoid general trend comparisons. Specify clear stats or metrics and get creative with them."
    )

    completion = openai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return completion.choices[0].message.content

def initialize_game_state(hometeam, awayteam):
    return {
        "team_stats": {
            hometeam: defaultdict(lambda: {
                "points": 0,
                "rebounds": 0,
                "assists": 0,
                "turnovers": 0,
                "rebounds": 0,
                "timeouts_taken": 0,
                "turnovers": 0,
            }),
            awayteam: defaultdict(lambda: {
                "points": 0,
                "rebounds": 0,
                "assists": 0,
                "turnovers": 0,
                "rebounds": 0,
                "timeouts_taken": 0,
                "turnovers": 0,
            }),
        },
        "player_stats": defaultdict(lambda: defaultdict(lambda: {
            "points": 0,
            "rebounds": 0,
            "assists": 0,
            "turnovers": 0,
            "two_pointers_attempted": 0,
            "two_pointers_made": 0,
            "three_pointers_attempted": 0,
            "three_pointers_made": 0,
            "free_throws_attempted": 0,
            "free_throws_made": 0,
            "shots_blocked": 0,
            "own_shots_have_been_blocked": 0,
            "defensive_rebounds": 0,
            "offensive_rebounds": 0,
            "charges_taken": 0,
            "personal_fouls": 0,
            "shooting_fouls": 0,
            "loose_ball_foul": 0,
            "steals": 0,
            "bad_passes": 0,
            "turnovers": 0,
    }))
}

def build_game_stats(play_data, game_stats, hometeam, awayteam):
    def parse_play_text(play_text):
        return_obj = {}
        if "blocks" in play_text:
            shot_type = "blocked_shot"
            blocker, shooter = play_text.split(" blocks ")
            shooter = shooter.split("'s")[0].strip()
            return_obj = {"shot_type": shot_type, "blocker": blocker, "shooter": shooter}
        elif 'defensive rebound' in play_text:
            rebounder = play_text.split(" defensive")[0]
            return_obj = {"shot_type": "rebound_defensive", "rebounder": rebounder}
        elif 'offensive rebound' in play_text:
            rebounder = play_text.split(" offensive")[0]
            return_obj = {"shot_type": "rebound_offensive", "rebounder": rebounder}
        elif 'team rebound' in play_text:
            team = play_text.split(" team")[0]
            return_obj = {"shot_type": "rebound_team", "rebounder": team}
        elif 'misses' in play_text:
            if 'free throw' in play_text:
                shot_type = "missed_free_throw"
            elif "three point" in play_text:
                shot_type = "missed_three_pointer"
            else:
                shot_type = "missed_two_pointer"
            shooter = play_text.split(" misses")[0]
            return_obj = {"shot_type": shot_type, "shooter": shooter}
        # elif 'personal foul' in play_text:
        #     shot_type = 'personal_foul'
        #     fouler = play_text.split(" personal")[0]
        #     return {"shot_type": shot_type, "fouler": fouler}
        elif "charge" in play_text:
            charger = play_text.split(" charge")[0]
            return_obj = {"shot_type": "charge", "charger": charger}
        elif "foul" in play_text:
            foul_type = play_text.split("foul")[0].strip().split()[-1]  
            fouler = play_text.split(" foul")[0]  
            return_obj = {"shot_type": foul_type + "_foul", "fouler": fouler}
        elif "steals" in play_text:
            stealer = play_text.split(" steals")[0].split("(")[-1].strip() 
            passer = play_text.split(" bad pass")[0].strip() 
            return_obj = {"shot_type": "steal", "stealer": stealer, "passer": passer}
        elif "turnover" in play_text:
            turnover_player = play_text.split("turnover")[0].strip()
            return_obj = {"shot_type": "turnover", "turnover_player": turnover_player}
        elif "makes" in play_text:
            if "free throw" in play_text:
                points = 1
                shot_type = "free_throw"
            elif "three point" in play_text:
                points = 3
                shot_type = "three_pointer"
            else:
                points = 2
                shot_type = "two_pointer"
            shooter = play_text.split(" makes")[0]
            return_obj = {"shot_type": shot_type, "points": points, "scorer": shooter}
        elif "enters the game" in play_text:
            new_player = play_text.split(" enters")[0].strip()
            old_player = play_text.split(" for")[0].split("(")[-1].strip()
            return_obj = {"shot_type": "substitution", "new_player": new_player, "old_player": old_player}
        elif 'timeout' in play_text:
            team = play_text.split(" timeout")[0]
            return_obj = {"shot_type": "timeout", "team": team}
        elif 'vs.' in play_text:
            player1 = play_text.split(" vs. ")[0]
            player2 = play_text.split(" vs. ")[1].split(" ")[0]
            return_obj = {"shot_type": "jump_ball", "player1": player1, "player2": player2}
        elif 'CHALLENGE' in play_text:
            team = play_text.split("]")[0].split("[")[-1]
            return_obj = {"shot_type": "challenge", "team": team}
        elif "End of " in play_text:
            return_obj = {"shot_type": "end_of_quarter"}
        elif "traveling" in play_text:
            traveler = play_text.split(" traveling")[0]
            return_obj = {"shot_type": "traveling", "traveler": traveler}
        elif "REVIEW" in play_text:
            review_team = play_text.split("]")[0].split("[")[-1]
            return_obj = {"shot_type": "review", "review_team": review_team}

        if "assists" in play_text:
            assisted_by = play_text.split("assists")[0].split("(")[-1].strip()
            return_obj["assister"] = assisted_by

        return return_obj        

    def update_stats(play, stats, hometeam, awayteam):
        homeAway = play.get("homeAway")
        quarter = play["period"]["number"]
        
        parsed_play = parse_play_text(play["text"])
        if parsed_play == {}:
            print(play["text"])

        if parsed_play['shot_type'] == "free_throw" or parsed_play['shot_type'] == "two_pointer" or parsed_play['shot_type'] == "three_pointer":
            points, scorer, shot_type = parsed_play.get("points"), parsed_play.get("scorer"), parsed_play.get("shot_type")
            
            #update team points
            if homeAway == "home":
                stats["team_stats"][hometeam][quarter]["points"] += points
            else:
                stats["team_stats"][awayteam][quarter]["points"] += points
            if scorer:
                stats["player_stats"][scorer][quarter]["points"] += points
                if shot_type == "two_pointer":
                    stats["player_stats"][scorer][quarter]["two_pointers_attempted"] += 1
                    stats["player_stats"][scorer][quarter]["two_pointers_made"] += 1 if points == 2 else 0
                elif shot_type == "three_pointer":
                    stats["player_stats"][scorer][quarter]["three_pointers_attempted"] += 1
                    stats["player_stats"][scorer][quarter]["three_pointers_made"] += 1 if points == 3 else 0
                elif shot_type == "free_throw":
                    stats["player_stats"][scorer][quarter]["free_throws_attempted"] += 1
                    stats["player_stats"][scorer][quarter]["free_throws_made"] += 1 if points == 1 else 0
        elif parsed_play['shot_type'] == "blocked_shot":
            blocker, shooter = parsed_play.get("blocker"), parsed_play.get("shooter")
            stats["player_stats"][blocker][quarter]["shots_blocked"] += 1
            stats["player_stats"][shooter][quarter]["own_shots_have_been_blocked"] += 1
            stats["player_stats"][shooter][quarter]["two_pointers_attempted"] += 1

        elif parsed_play['shot_type'] == "rebound_defensive":
            rebounder = parsed_play.get("rebounder")
            stats["player_stats"][rebounder][quarter]["defensive_rebounds"] += 1
        elif parsed_play['shot_type'] == "rebound_offensive":
            rebounder = parsed_play.get("rebounder")
            stats["player_stats"][rebounder][quarter]["offensive_rebounds"] += 1
        elif parsed_play['shot_type'] == "rebound_team":
            rebounder = parsed_play.get("rebounder")
            if homeAway == "home":
                stats["team_stats"][hometeam][quarter]["rebounds"] += 1
            else:
                stats["team_stats"][awayteam][quarter]["rebounds"] += 1
        elif parsed_play['shot_type'] == "missed_free_throw":
            shooter = parsed_play.get("shooter")
            stats["player_stats"][shooter][quarter]["free_throws_attempted"] += 1
        elif parsed_play['shot_type'] == "missed_three_pointer":
            shooter = parsed_play.get("shooter")
            stats["player_stats"][shooter][quarter]["three_pointers_attempted"] += 1
        elif parsed_play['shot_type'] == "missed_two_pointer":
            shooter = parsed_play.get("shooter")
            stats["player_stats"][shooter][quarter]["two_pointers_attempted"] += 1
        elif parsed_play['shot_type'] == "charge":
            charger = parsed_play.get("charger")
            stats["player_stats"][charger][quarter]["charges_taken"] += 1
        elif parsed_play['shot_type'] == "personal_foul":
            fouler = parsed_play.get("fouler")
            stats["player_stats"][fouler][quarter]["personal_fouls"] += 1
        elif parsed_play['shot_type'] == 'shooting_foul':
            fouler = parsed_play.get("fouler")
            stats["player_stats"][fouler][quarter]["shooting_fouls"] += 1
        elif parsed_play['shot_type'] == 'ball_foul':
            fouler = parsed_play.get("fouler")
            stats["player_stats"][fouler][quarter]["loose_ball_foul"] += 1
        elif parsed_play['shot_type'] == 'steal':
            stealer = parsed_play.get("stealer")
            passer = parsed_play.get("passer")
            stats["player_stats"][stealer][quarter]["steals"] += 1
            stats["player_stats"][passer][quarter]["bad_passes"] += 1
        elif parsed_play['shot_type'] == 'turnover':
            turnover_player = parsed_play.get("turnover_player")
            stats["player_stats"][turnover_player][quarter]["turnovers"] += 1

            if homeAway == "home":
                stats["team_stats"][hometeam][quarter]["turnovers"] += 1
            else:
                stats["team_stats"][awayteam][quarter]["turnovers"] += 1
        elif parsed_play['shot_type'] == 'timeout':
            team = parsed_play.get("team")
            if homeAway == "home":
                stats["team_stats"][hometeam][quarter]["timeouts_taken"] += 1
            else:
                stats["team_stats"][awayteam][quarter]["timeouts_taken"] += 1

        if "assister" in parsed_play:
            assister = parsed_play.get("assister")
            stats["player_stats"][assister][quarter]["assists"] += 1
            if homeAway == "home":
                stats["team_stats"][hometeam][quarter]["assists"] += 1
            else:
                stats["team_stats"][awayteam][quarter]["assists"] += 1


    for play in play_data:
        update_stats(play, game_stats, hometeam, awayteam)

    return game_stats


def extract_player_names(play_text, hometeam, awayteam):
    if hometeam in play_text:
        return [hometeam]
    elif awayteam in play_text:
        return [awayteam]

    if "End of " in play_text:
        return play_text
    
    if "turnover"  in play_text:
        return play_text
    
    # Use regex to match 'Firstname Lastname' patterns for player names
    name_pattern = r"([A-Z][a-z]+(?: [A-Z][a-z]+)+)"  # Handles multiple name parts like 'Lindy Waters III'
    matches = re.findall(name_pattern, play_text)
    if matches:
        return matches

    name_pattern = r"([A-Z][a-z]+(?: [A-Z][a-z]+|[A-Z]\.)+)"
    matches = re.findall(name_pattern, play_text)
    if matches:
        return matches
    
    name_pattern = r"([A-Z][a-z]*[A-Z][a-z]*(?: [A-Z][a-z]+)+)"
    matches = re.findall(name_pattern, play_text)
    if matches:
        return matches
    

def make_hashable(item):
    if isinstance(item, dict):
        return frozenset((key, make_hashable(value)) for key, value in item.items())
    elif isinstance(item, list):
        return tuple(make_hashable(x) for x in item)
    return item  

def find_new_dicts(list_b, play_map):
    returnlist = []
    for i in list_b:
        hashed = make_dict_hashable(i)
        if hashed not in play_map:
            play_map[hashed] = 1
            returnlist.append(i)
    return returnlist, play_map

#convert dict to string
def make_dict_hashable(d, cache={}):
    return json.dumps(d, sort_keys=True) 

In [158]:
# FIRST ATTEMPT AT GETTING PLAY BY PLAY DATA
# with urllib.request.urlopen(url) as response:
#     html = response.read()

# soup = BeautifulSoup(html, 'html.parser')
# script_tag = soup.find('script', text=lambda x: x and 'playGrps' in x)
# script_content = script_tag.string

# pattern = r'(\{"playGrps.*?}}]])'
# match = re.search(pattern, script_content, re.DOTALL)

# if match:
#     extracted_text = match.group(1)
#     extracted_text += "}"
#     raw_play_data = json.loads(extracted_text)

# number_of_quarters = len(raw_play_data['playGrps'])
# print(f"number of quarters happened/happening in the game: {number_of_quarters}")

# for quarter in range(0, number_of_quarters):
#     print(f"Quarter: {quarter+1}")
#     quarter_play_data = raw_play_data['playGrps'][quarter]
    
    # Locate the div or section containing the play-by-play data for the current quarter
    # quarter_section = soup.find('div', class_='ResponsiveTable playByPlay__table playByPlay__table--default')  # Adjust the id based on the actual HTML structure
    # if quarter_section:
    #     # Find all the rows representing each play
    #     play_rows = quarter_section.find_all('tr', class_='playByPlay__tableRow Table__TR Table__TR--sm Table__even')  # Adjust class names as needed
    #     for row in play_rows:
    #         # Extract the time, play description, and score
    #         time = row.find('td', class_='playByPlay__time Table__TD').text  # Replace 'time-stamp-class' with the correct class
    #         play = row.find('td', class_='playByPlay__text tl Table__TD').text  # Replace with the correct class
    #         score_mia = row.find('td', class_='playByPlay__score playByPlay__score--away tr Table__TD').text  # Replace with the correct class
    #         score_bos = row.find('td', class_='playByPlay__score playByPlay__score--home tr Table__TD').text  # Replace with the correct class
            
    #         play_info = {'time': time, 'play': play}
    #         play_data.append(play_info)

#### DRIVER

In [206]:

def find_difference(str1, str2):
    # Create a SequenceMatcher object
    matcher = SequenceMatcher(None, str1, str2)
    
    # Iterate over the matched blocks
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            # If strings aren't equal, print the part where they differ
            diff1 = str1[i1:i2]
            diff2 = str2[j1:j2]
            return (diff1, diff2)  # Returns the differing parts

    # If strings are equal
    return None

In [ ]:
## BACKUP BACKUP
# url = 'https://www.espn.com/nba/playbyplay/_/gameId/401716985'
url = 'https://www.espn.com/nba/playbyplay/_/gameId/401704629'
# url = 'http://127.0.0.1:5001/nba/playbyplay/401716985'

async def fetch_data(session):
    async with session.get(url) as response:
        html = await response.text()
        return html

game_info = {
    'team1': 'Indiana Pacers',
    'team2': 'Detroit Pistons',
    'date': 'October 23, 2024',
}

stop_event = asyncio.Event()
global_compound_queries = set()
GAME_STATS = initialize_game_state(game_info['team1'], game_info['team2'])

async def extract_play_data():
    last_raw_play_data = None  # Cache for the last fetched data
    key_moments = []
    key_moment_length_since_last_query = 0
    play_map = {}
    prev_length = 0
    async with aiohttp.ClientSession() as session:
        while not stop_event.is_set():
            html = await fetch_data(session)
            
            pattern = r'(\{"playGrps.*?}}]])'
            match = re.search(pattern, html, re.DOTALL)
            
            if match:
                extracted_text = match.group(1) + "}"
                raw_play_data = json.loads(extracted_text)['playGrps']
                raw_all_plays = []
                for quarter in raw_play_data:
                    raw_all_plays.extend(quarter)
                
                # if len(play_map) > 0:

                #     # new_plays = [i for i in all_raw if i not in all_previous]
                #     # new_plays_dict = find_new_dicts(last_raw_play_data, raw_play_data)
                #     new_plays_dict, play_map = find_new_dicts(raw_all_plays, play_map)
                #     if len(new_plays_dict) > 0:
                #         print("----NEW PLAY----")
                #         print(str([i['text'] for i in new_plays_dict]))
                #         key_moments_current = process_play_by_play(new_plays_dict) 
                #         key_moments.extend(key_moments_current)

                #         global GAME_STATS
                #         GAME_STATS = build_game_stats(new_plays_dict, GAME_STATS, game_info['team1'], game_info['team2'])

                #         if len(key_moments) % 5 == 0 and len(key_moments) > key_moment_length_since_last_query:
                #             print("querying")
                #             key_moment_length_since_last_query = len(key_moments)

                #             start = time.time()
                #             compound_queries = generate_queries_with_gpt(key_moments, game_info, open_ai_key) 
                #             end_time = time.time()
                #             elapsed_time = end_time - start
                #             print(f"Time taken to generate queries: {elapsed_time:.6f} seconds")

                #             compound_queries = compound_queries.strip().split('\n')
                #             compound_queries = [q.strip() for q in compound_queries if q.strip()]
                #             compound_queries = [re.sub(r'^\d+\.\s*', '', q.strip()) for q in compound_queries if q.strip()]
                            
                #             # print("----COMPOUND QUERIES----")
                #             # print(compound_queries)

                #             for cq in compound_queries: 
                #                 if cq not in global_compound_queries:
                #                     print("---QUERY: " + cq)
                #                     # context = fetch_relevant_game_stats(cq, game_stats)
                #                     # print(context)
                #                     thelist = list(agent_executor.stream({"input": cq}))
                #                     print("----ANSWER---- " + thelist[-1]['output'])

                                    
                #                     print("------------------------------------------------------------------")
                #                     # agent_executor.run({"input": cq})
                #                     # break
                #             global_compound_queries.update(compound_queries)


                # else:
                #     for q in raw_play_data:
                #         for i in q:
                #             play_map[make_dict_hashable(i)] = 1
            

            await asyncio.sleep(1)  # Poll every 5 seconds

# Run the coroutine in an existing event loop
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.create_task(extract_play_data())


In [13]:
id_to_play = {}
count_to_id = {}
id_to_play_info = {}
num_plays = 0

game_id = 401704663
# websocket.enableTrace(True)

last_raw_play_data = None  # Cache for the last fetched data
key_moments = []
key_moment_length_since_last_query = 0
play_map = {}
prev_length = 0
temp_count = 0


current_window_of_plays = {}

game_info = {
    'team1': 'Indiana Pacers', #home
    'team2': 'Philadelphia 76ers', #away
    'date': 'October 23, 2024',
}

home_team_id = id_to_team[game_info['team1']]
away_team_id = id_to_team[game_info['team2']]

GAME_STATS = initialize_game_state(game_info['team1'], game_info['team2'])


def on_message(ws, message):
    # print("Socket message received from server:")
    data = json.loads(message)

    #initiator messages
    if data.get("op") == "C" and data.get("rc") == 200:
        #send the S messages
        sid = data.get("sid")
        print("retrieved session id " + sid)
        msg = {
            "op": "S",
            "sid": sid,
            "tc": f"gp-basketball-nba-{game_id}"
        }
        ws.send(json.dumps(msg))

        msg = {
            "op": "S",
            "sid": sid,
            "tc": "event-basketball-nba"
        }
        ws.send(json.dumps(msg))

    if data.get('pl') != None:
        try:
            pl = json.loads(data.get('pl'))
            if type(pl) == dict:
                pl = pl.get('pl')
                compressed_raw = base64.b64decode(pl)
                decompressed_data = zlib.decompress(compressed_raw)
                result = json.loads(decompressed_data)
                pattern = r'^/plays/\d+/text$'
                if type(result) == list:
                    for r in result:
                        play_occur = False
                        if 'op' in r and r['op'] == 'add' and 'path' in r and r['path'] == '/plays/-':
                            play_text = r['value']['text']
                            play_id = r['value']['id']
                            play_clock = r['value']['clock']
                            play_period = r['value']['period']
                            play_home_score = r['value']['homeScore']
                            play_away_score = r['value']['awayScore']

                            team_id = r['value']['team']['id']
                            if team_id == home_team_id:
                                play_home_away = 'home'
                            else:
                                play_home_away = 'away'

                            global num_plays

                            play_count = num_plays + 1
                            num_plays += 1

                            id_to_play[play_id] = play_text
                            count_to_id[play_count] = play_id
                            # print(r)

                            print("----NEW PLAY----")
                            print(play_text)
                            play_occur = True
                            
                        # elif 'op' in r and r['op'] == 'replace' and 'path' in r and re.match(pattern, r['path']):
                        #     #split on /
                        #     splits = r['path'].split('/')
                        #     play_count = splits[2]
                        #     play_text = r['value']

                        #     global num_plays
                        #     global temp_count

                        #     if play_count in count_to_id:
                        #         play_id = count_to_id[play_count]
                        #         id_to_play[play_id] = play_text
                        #     else:
                        #         play_count = num_plays + 1
                        #         num_plays += 1

                                
                        #         play_id = temp_count + 1
                        #         temp_count += 1

                        #         id_to_play[play_id] = play_text
                        #         count_to_id[play_count] = play_id
                            
                            
                        #     print("----MODIFIED PLAY----")
                        #     print(play_text)
                        #     play_occur = True

                        if play_occur:
                            global current_window_of_plays
                            if play_id not in current_window_of_plays:
                                newdict = {"text": play_text, "id": play_id, "clock": play_clock, "period": play_period, "homeAway": play_home_away, "homeScore": play_home_score, "awayScore": play_away_score}
                                current_window_of_plays[play_id] = newdict
                            else:
                                current_window_of_plays[play_id]["text"] = play_text

                        

                        if len(current_window_of_plays) == 5:
                            key_moments_current = process_play_by_play(list(current_window_of_plays.values())) 
                            key_moments.extend(key_moments_current)

                            global GAME_STATS
                            GAME_STATS = build_game_stats(list(current_window_of_plays.values()), GAME_STATS, game_info['team1'], game_info['team2'])

                        
                            if len(key_moments) % 5 == 0 and len(key_moments) > key_moment_length_since_last_query:
                                print("querying")
                                # key_moment_length_since_last_query = len(key_moments)

                                # compound_queries = generate_queries_with_gpt(key_moments, game_info, open_ai_key)

                                # compound_queries = compound_queries.strip().split('\n')
                                # compound_queries = [q.strip() for q in compound_queries if q.strip()]
                                # compound_queries = [re.sub(r'^\d+\.\s*', '', q.strip()) for q in compound_queries if q.strip()]

                                # for cq in compound_queries:
                                #     if cq not in global_compound_queries:
                                #         print("---QUERY: " + cq)
                                #         thelist = list(agent_executor.stream({"input": cq}))
                                #         print("----ANSWER---- " + thelist[-1]['output'])
                                #         print("------------------------------------------------------------------")

                                # global_compound_queries.update(compound_queries)

                            current_window_of_plays = {}

        except json.JSONDecodeError:
            pass


# Function to handle errors
def on_error(ws, error):
    print("Error occurred:", error)
    # traceback.print_exc()

# Function to handle WebSocket closure
def on_close(ws, close_status_code, close_msg):
    print("Connection closed:", close_status_code, close_msg)

# Function to handle WebSocket opening
def on_open(ws):
    print("Socket open.")
    # Send the initial message after opening the socket
    initial_message = '{"op": "C"}'
    ws.send(initial_message)

# Main function to establish WebSocket connection
def run_websocket():
    # Fetch the WebSocket URL
    # fetch_url = 'https://fastcast.semfs.engsvc.go.com/public/websockethost'
    # response = requests.get(fetch_url)

    # if response.status_code != 200:
    #     print('Looks like there was a problem. Status Code:', response.status_code)
    #     return

    # data = response.json()
    # ws_uri = f"wss://{data['ip']}:{data['securePort']}/FastcastService/pubsub/profiles/12000?TrafficManager-Token={data['token']}"
    # print("WebSocket URL:", ws_uri)

    ws_uri = "wss://pwb291c4a2-4e74-42e1-985b-6d705c054b3a-54-200-120-101.fastcast.semfs.engsvc.go.com:9573/FastcastService/pubsub/profiles/12000?TrafficManager-Token=MTczMDA2NDA5MzM3OA==:gImBL7EaVSbvVzcZQfetVKyo/DA="

    # Create the WebSocket connection
    ws = websocket.WebSocketApp(
        ws_uri,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )

    # Run the WebSocket connection
    ws.run_forever()

# Start the WebSocket client in a separate thread
ws_thread = threading.Thread(target=run_websocket)
ws_thread.start()

# Allow the user to stop the client gracefully
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping WebSocket client...")
    ws_thread.join()  # Wait for the WebSocket thread to finish


Socket open.
retrieved session id 45f37108233047d685cf6184ff6d3f60
----NEW PLAY----
Ben Sheppard enters the game for Tyrese Haliburton
----NEW PLAY----
Tyrese Maxey blocks T.J. McConnell 's 14-foot shot
----NEW PLAY----
Pacers offensive team rebound
Error occurred: argument should be a bytes-like object or ASCII string, not 'list'
Stopping WebSocket client...
----NEW PLAY----
Myles Turner misses shot
----NEW PLAY----
Pacers offensive team rebound
----NEW PLAY----
Pacers offensive team rebound


KeyboardInterrupt: 

In [229]:
# Function to stop the running event loop
def stop_running_loop():
    stop_event.set()

stop_running_loop()


In [ ]:

def fetch_connection_info():
    url = 'https://fastcast.semfs.engsvc.go.com/public/websockethost'
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to fetch connection info. Status Code: {response.status_code}")


session_id = "04ca23d0-9250-11ef-90a7-6ecccb26423b"
profile_id = "2e372edc-efe0-4e43-987c-cd2460542587"


from datetime import datetime, timezone
import websocket

# secondary_ws_url = "wss://espn.connections.edge.bamgrid.com/66990c/connection?X-Application-Version=0.0.1&X-BAMSDK-Client-ID=espn-a9b93989&X-BAMSDK-Platform=javascript/macosx/chrome&X-BAMSDK-Version=27.1&X-Request-ID=&X-Request-Id=4b4a0654-f962-4fd1-ac66-9228ff5ec9f1"
connection_info = fetch_connection_info()
secondary_ws_url = f"wss://{connection_info['ip']}:{connection_info['securePort']}/FastcastService/pubsub/profiles/12000?TrafficManager-Token={connection_info['token']}"
    

def on_open_secondary(ws):
    print("Secondary WebSocket connection opened.")

    connection_info = fetch_connection_info()
    print(connection_info)
    # Create and send the authentication payload similar to the website
    auth_payload = {
        "datacontenttype": "application/json;charset=utf-8",
        "id": "6625c9c9-3514-440a-8ef7-81c779293b4f",  # Use a UUID or unique identifier
        "schemaurl": "https://github.bamtech.co/schema-registry/schema-registry/blob/master/dss/event/edge/1.0.0/sdk/authentication.oas2.yaml",
        "source": "urn:dss:source:sdk:javascript:macosx:chrome",
        "subject": f"sessionId={session_id},profileId={profile_id}",
        "time": datetime.now(timezone.utc).isoformat(),
        "type": "urn:dss:event:edge:sdk:authentication",
        "data":{
            "accessToken": connection_info["token"]  
        }
    }
    ws.send(json.dumps(auth_payload))
    print("Sent authentication payload to secondary connection:", auth_payload)

def on_message_secondary(ws, message):
    print("Received message on secondary connection:", message)

def on_error_secondary(ws, error):
    print("Error on secondary connection:", error)

def on_close_secondary(ws, close_status_code, close_msg):
    print(f"Secondary WebSocket connection closed with status code: {close_status_code}, message: {close_msg}")

# Function to run the secondary WebSocket connection
def run_secondary_ws():
    ws_secondary = websocket.WebSocketApp(
        secondary_ws_url,
        on_open=on_open_secondary,
        on_message=on_message_secondary,
        on_error=on_error_secondary,
        on_close=on_close_secondary,
        subprotocols=["vnd.dss.edge+json"]
    )
    ws_secondary.run_forever()

# Run the secondary WebSocket connection independently
run_secondary_ws()

In [ ]:



id_to_play = {}
count_to_id = {}
num_plays = 0
gjiro = 0

#schema:
#id, count, text
#id will be the store val because each add will contain this
#count will be a running count for each
#when a replace is encountered, we search pased on the count and replace the text


# Function to handle incoming WebSocket messages
def on_message(ws, message):
    # print("Socket message received from server:")
    data = json.loads(message)

    if data.get("op") == "C" and data.get("rc") == 200:
        #send the S messages
        sid = data.get("sid")
        print("retrieved session id " + sid)
        msg = {
            "op": "S",
            "sid": sid,
            "tc": "gp-basketball-nba-401704658"
        }
        ws.send(json.dumps(msg))

        msg = {
            "op": "S",
            "sid": sid,
            "tc": "event-basketball-nba"
        }
        ws.send(json.dumps(msg))

    if data.get('pl') != None:
        try:
            pl = json.loads(data.get('pl'))
            if type(pl) == dict:
                pl = pl.get('pl')
                compressed_raw = base64.b64decode(pl)
                decompressed_data = zlib.decompress(compressed_raw)
                result = json.loads(decompressed_data)
                pattern = r'^/plays/\d+/text$'
                if type(result) == list:
                    for r in result:
                        if 'op' in r and r['op'] == 'add' and 'path' in r and r['path'] == '/plays/-':
                            play_text = r['value']['text']
                            play_id = r['value']['id']

                            play_count = num_plays + 1
                            num_plays += 1

                            id_to_play[play_id] = play_text
                            count_to_id[play_count] = play_id
                            print(r)
                            
                        elif 'op' in r and r['op'] == 'replace' and 'path' in r and re.match(pattern, r['path']):
                            play_count = int(re.match(pattern, r['path']).group(1))
                            id_of_play_to_replace = count_to_id[play_count]

                            play_text = r['value']

                            id_to_play[id_of_play_to_replace] = play_text
                            print(r)

                        print(play_text)

        except json.JSONDecodeError:
            pass


# Function to handle errors
def on_error(ws, error):
    print("Error occurred:", error)

# Function to handle WebSocket closure
def on_close(ws, close_status_code, close_msg):
    print("Connection closed:", close_status_code, close_msg)

# Function to handle WebSocket opening
def on_open(ws):
    print("Socket open.")
    # Send the initial message after opening the socket
    initial_message = '{"op": "C"}'
    ws.send(initial_message)

# Main function to establish WebSocket connection
def run_websocket():
    # Fetch the WebSocket URL
    # fetch_url = 'https://fastcast.semfs.engsvc.go.com/public/websockethost'
    # response = requests.get(fetch_url)

    # if response.status_code != 200:
    #     print('Looks like there was a problem. Status Code:', response.status_code)
    #     return

    # data = response.json()
    # ws_uri = f"wss://{data['ip']}:{data['securePort']}/FastcastService/pubsub/profiles/12000?TrafficManager-Token={data['token']}"
    # print("WebSocket URL:", ws_uri)

    ws_uri = "wss://pwc774c190-55c1-4d7d-8b2b-f39a0e6ae077-34-219-67-59.fastcast.semfs.engsvc.go.com:9573/FastcastService/pubsub/profiles/12000?TrafficManager-Token=MTcyOTk5MzEyNDkyNg==:YVsjedNc5JT838MGapqN2qXkwUQ="

    # Create the WebSocket connection
    ws = websocket.WebSocketApp(
        ws_uri,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )

    # Run the WebSocket connection
    ws.run_forever()

# Start the WebSocket client in a separate thread
ws_thread = threading.Thread(target=run_websocket)
ws_thread.start()

# Allow the user to stop the client gracefully
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping WebSocket client...")
    ws_thread.join()  # Wait for the WebSocket thread to finish


In [ ]:
#NON LOOP VERSION OF POLLING, FIRST ATT RUNNING SEQ OF EVENTS
url = 'https://www.espn.com/nba/playbyplay/_/gameId/401716985'

def fetch_live_play_by_play(url):

    with urllib.request.urlopen(url) as response:
        html = response.read()

    soup = BeautifulSoup(html, 'html.parser')
    script_tag = soup.find('script', string=lambda x: x and 'playGrps' in x)
    script_content = script_tag.string

    pattern = r'(\{"playGrps.*?}}]])'
    match = re.search(pattern, script_content, re.DOTALL)

    if match:
        extracted_text = match.group(1)
        extracted_text += "}"
        raw_play_data = json.loads(extracted_text)

    number_of_quarters = len(raw_play_data['playGrps'])
    # print(f"number of quarters happened/happening in the game: {number_of_quarters}")

    all_game_play_data = []

    for quarter in range(0, number_of_quarters):
        # print(f"Quarter: {quarter+1}")
        quarter_play_data = raw_play_data['playGrps'][quarter]

        all_game_play_data.extend(quarter_play_data)

    is_game_ended = False
    if number_of_quarters == 4 and all_game_play_data[-1]['text'] == 'End of Game':
        is_game_ended = True

    return all_game_play_data, is_game_ended


def is_new_plays_occur(current_events, previous_events):
    return current_events != previous_events


def fetch_relevant_game_stats(query, game_stats):
    # Check for player in the query
    player_match = re.search(r"(?P<player>\w+\s\w+)'s", query)
    relevant_stats = {}

    if player_match:
        player_name = player_match.group("player")
        print(player_name)
        player_stats = game_stats.get("player_stats", {}).get(player_name)
        if player_stats:
            # Check if shooting percentage is requested
            if "shooting percentage" in query.lower():
                # points = sum(q["points"] for q in player_stats.values())
                shots_taken = sum(q["two_pointers_attempted"] for q in player_stats.values()) + sum(q["three_pointers_attempted"] for q in player_stats.values())
                shots_made = sum(q["two_pointers_made"] for q in player_stats.values()) + sum(q["three_pointers_made"] for q in player_stats.values())
                
                shooting_percentage = (shots_made / shots_taken) * 100 if shots_taken else 0
                relevant_stats["shooting_percentage"] = shooting_percentage

            # Add other stats as needed
            if "points" in query.lower():
                total_points = sum(q["points"] for q in player_stats.values())
                relevant_stats["points"] = total_points

            # Add per-quarter stats if the query references quarters
            if "quarter" in query.lower():
                relevant_stats["quarter_stats"] = player_stats

    return relevant_stats
    
global_game_stats = None
def poll_for_updates(url, game_info, interval=2):
    previous_events = []
    previous_key_moments = []
    while True:
        print("polling for updates in game")
        plays_so_far, is_game_ended = fetch_live_play_by_play(url) 
        print(plays_so_far)
        if is_new_plays_occur(plays_so_far, previous_events):  # Detect new play
            num_new_plays = len(plays_so_far) - len(previous_events)
            new_plays = plays_so_far[-num_new_plays:]
            #print text of those new plays
            # print("new plays: " + str([play['text'] for play in new_plays]))    
            print(len(plays_so_far))

            print([play['text'] for play in plays_so_far])
            print("new play occurred: " + str(new_plays))
            key_moments = process_play_by_play(plays_so_far) 

            game_stats = build_game_stats(plays_so_far, game_info['team1'], game_info['team2'])
            global global_game_stats
            global_game_stats = game_stats  
            

            # query = "What is Stephen Curry's shooting percentage in this game compared to his season average?"
            # player_name =  extract_player_names(query, game_info['team1'].split()[-1], game_info['team2'].split()[-1])[0]
            # player_stats = game_stats.get("player_stats", {}).get(player_name)
            # print("player stat information: ")
            # print(json.dumps(dict(global_game_stats), indent=4))

            # context = fetch_relevant_game_stats(query, game_stats)
            # print(context)


            if is_new_plays_occur(key_moments, previous_key_moments):
                print("new key moment")
                # compound_queries = generate_queries_with_gpt(key_moments, game_info, open_ai_key) 
                # compound_queries = compound_queries.strip().split('\n')
                # compound_queries = [q.strip() for q in compound_queries if q.strip()]
                # compound_queries = [re.sub(r'^\d+\.\s*', '', q.strip()) for q in compound_queries if q.strip()]
                
                # for cq in compound_queries: 
                #     print("QUERY: " + cq)
                #     # context = fetch_relevant_game_stats(cq, game_stats)
                #     # print(context)
                #     list(agent_executor.stream({"input": cq}))
                #     print("------------------------------------------------------------------")
                #     # agent_executor.run({"input": cq})
                #     # break

        previous_events = plays_so_far
        previous_key_moments = key_moments

        if is_game_ended:
            print("game has ended")
            break

        time.sleep(interval)  # Poll every 5 seconds

game_info = {
    'team1': 'Golden State Warriors',
    'team2': 'Sacramento Kings',
    'date': 'October 11, 2024',
}

poll_for_updates(url, game_info)




process the key parts of the play data

answering the query and getting the stat info

In [150]:

def extract_player_name_from_query(query):
    name_pattern = r"([A-Z][a-z]+ [A-Z][a-z]+)"
    matches = re.findall(name_pattern, query)
    
    if matches:
        return matches[0]
    
    return None

def extract_team_name_from_query(query, known_teams):
    query_lower = query.lower()    
    for team in known_teams:
        if team.lower() in query_lower:
            return team  # Return the first matched team name
    
    return None

def query_game_stats(query):
    team_name = extract_team_name_from_query(query, [game_info['team1'], game_info['team2']])  
    if team_name:
        return GAME_STATS.get('team_stats', {}).get(team_name, {})
    
    player_name = extract_player_name_from_query(query)  
    if player_name:
        player_stats = GAME_STATS.get('player_stats', {}).get(player_name, {})
        return player_stats if player_stats else "Player not found"
    
    return "No relevant game stats found for this query."

In [163]:

def search_statmuse(query: str) -> str:
  URL = f'https://www.statmuse.com/nba/ask/{query}'
  page = requests.get(URL)
  
  soup = BeautifulSoup(page.content, "html.parser")
  return soup.find("div", class_="flex flex-col justify-between @lg/hero:items-start").text

search_out = search_statmuse("Who is the highest scoring player on the Los Angeles Lakers of all time")
print(search_out)

statmuse_tool = Tool(
    name = "Statmuse",
    func = search_statmuse,
    description = "A sports search engine. Use this more than normal search if the question is about NBA basketball, like 'who is the highest scoring player in the NBA?'. Always specify a year or timeframe with your search. Only ask about one player or team at a time, don't ask about multiple players at once."
)

serpapi = SerpAPIWrapper(serpapi_api_key=serp_api_key)
serpapi_tool = Tool(
    name="SerpAPI",
    description="Use this tool to search the web for information.",
    func=serpapi.run
)

context_tool = Tool(
    name="Currentgame",
    description="Use this to get information/context about the current game and players in the current game",
    func=query_game_stats
)

  Kobe Bryant has put up the most career points for the Lakers, with 33,643 points.   


In [218]:

question="What is Stephen Curry's shooting percentage in this game compared to his season average so far?"

#determine if query is game-specific/related or not:


tools = [statmuse_tool, serpapi_tool, context_tool]

llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=open_ai_key)
llm_with_tools = llm.bind_tools(tools)

# tools = load_tools(["serpapi", "llm-math"], llm=llm, serpapi_api_key="d08e104f693e95d6dfa1194e4e560db5239643147352af8ed4881a9e5be5d7cd") + [statmuse_tool]

prompt = ChatPromptTemplate.from_messages(
    [
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)
list(agent_executor.stream({"input": question}))




[{'actions': [ToolAgentAction(tool='Currentgame', tool_input='Stephen Curry shooting percentage', log='\nInvoking: `Currentgame` with `Stephen Curry shooting percentage`\n\n\n', message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_ws0IGlSA2aePLyerFTAf0HfC', 'function': {'arguments': '{"__arg1": "Stephen Curry shooting percentage"}', 'name': 'Currentgame'}, 'type': 'function'}, {'index': 1, 'id': 'call_fUam5XWO1Eix1rSx7J7bF9pz', 'function': {'arguments': '{"__arg1": "Stephen Curry shooting percentage 2023-24 season"}', 'name': 'Statmuse'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_45c6de4934'}, id='run-10f4f0dc-11a9-4402-ab1f-7336ccb7bf56', tool_calls=[{'name': 'Currentgame', 'args': {'__arg1': 'Stephen Curry shooting percentage'}, 'id': 'call_ws0IGlSA2aePLyerFTAf0HfC', 'type': 'tool_call'}, {'name': 'Statmuse', 'args': {'__arg1': 'Stephen Curry shoot